In [ ]:
import httpx, pandas as pd, numpy as np, os
from dotenv import load_dotenv

load_dotenv("../.env")

CACHE = "history.parquet"
PAGE_SIZE = 1000
DEFCON_FIRST_SEASON = "2025-26" #defensive_contribution didn't exist before this, older 0s mean missing not zero

def fetch_all_gameweeks(c):
    #short page signals the end, same as app/player_client.py
    rows, offset = [], 0
    while True:
        r = c.get("/player-gameweeks", params={"limit": PAGE_SIZE, "offset": offset})
        r.raise_for_status()
        page = r.json()
        rows.extend(page)
        if len(page) < PAGE_SIZE:
            return rows
        offset += PAGE_SIZE

def build_history():
    headers = {"X-API-Key": os.environ["PLAYER_SERVICE_API_KEY"]}
    with httpx.Client(base_url="http://localhost:8000", headers=headers, timeout=30) as c:
        df = pd.DataFrame(fetch_all_gameweeks(c))

    df.loc[df["season"] < DEFCON_FIRST_SEASON, "defensive_contribution"] = np.nan
    return df

if os.path.exists(CACHE):
    df = pd.read_parquet(CACHE)
else:
    df = build_history()
    df.to_parquet(CACHE)